Extraindo PDF

In [206]:
!pip install transformers datasets accelerate sentencepiece PyMuPDF torch huggingface_hub python-dotenv



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [211]:
!pip install ipywidgets bitsandbytes

   ---------------------------------------- 0.0/55.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/55.4 MB ? eta -:--:--
   ---------------------------------------- 0.5/55.4 MB 1.8 MB/s eta 0:00:31
    --------------------------------------- 0.8/55.4 MB 1.4 MB/s eta 0:00:40
    --------------------------------------- 1.0/55.4 MB 1.5 MB/s eta 0:00:36
    --------------------------------------- 1.3/55.4 MB 1.2 MB/s eta 0:00:45
   - -------------------------------------- 1.6/55.4 MB 1.2 MB/s eta 0:00:45
   - -------------------------------------- 1.8/55.4 MB 1.2 MB/s eta 0:00:44
   - -------------------------------------- 2.1/55.4 MB 1.2 MB/s eta 0:00:44
   - -------------------------------------- 2.1/55.4 MB 1.2 MB/s eta 0:00:44
   - -------------------------------------- 2.1/55.4 MB 1.2 MB/s eta 0:00:44
   - -------------------------------------- 2.1/55.4 MB 1.2 MB/s eta 0:00:44
   - -------------------------------------- 2.1/55.4 MB 1.2 MB/s eta 0:00:44
   - --------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [212]:
import json
import re
# import pdfplumber
import fitz  # PyMuPDF
import re
import torch
from pathlib import Path
# from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

In [213]:
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# Sem login() — token vai direto nos métodos abaixo
print("Token carregado:", "✅" if HF_TOKEN else "❌ Token não encontrado no .env")

Token carregado: ✅


In [214]:

def extract_text_from_pdf_pymupdf(pdf_path, skip_pages=6):
    """Extrai texto de um arquivo PDF usando PyMuPDF."""
    pages = []
    
    # Abre o documento PDF
    doc = fitz.open(pdf_path)
    
    # Itera pelas páginas a partir do índice 'skip_pages'
    for page_num in range(skip_pages, len(doc)):
        page = doc[page_num]
        page_text = page.get_text() # Extrai o texto da página

        if page_text:
            pages.append(page_text)
            
    doc.close() # É uma boa prática fechar o documento
    return "\n".join(pages)

def clean_pdf_text(text):
    # Corrige palavras quebradas por hífen no final da linha
    text = re.sub(r'(\w+)-\s+(\w+)', r'\1\2', text) 

    # Remove múltiplas quebras de linha
    text = re.sub(r'\n+', '\n', text)

    # Remove múltiplos espaços e tabulações
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()

# Configuração do caminho do PDF
pdf_path = "20260018701.pdf"

# Execução
full_text = extract_text_from_pdf_pymupdf(pdf_path)
full_text = clean_pdf_text(full_text)

print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

Total de caracteres extraídos: 264093

--- INÍCIO DO TEXTO ---

A medicina da simplicidade
– trabalhar com plantas é a
ciência do simples
Alésio dos Passos Santos – ambientalista, cultivador e educador de plantas medicinais (com contribuições de César Paulo
Simionatto, Murilo Leandro Marcos, Leila Nery dos Santos Souza,
Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon
Bittencourt)
A medicina do sinergismo e a energia de cura
O sinergismo é uma das coisas mais importantes no mundo
das plantas. O remédio de farmácia, o que que é? Um princípio
ativo i


Dividindo em chunks

In [215]:
def dividir_em_paragrafos(texto):
    """
    Divide o texto em parágrafos significativos.
    """
    # Dividindo os paragrafos por quebra de linha e limpando espaços extras
    paragrafos = texto.split("\n")
    # Filtro de ruído: ignora fragmentos menores que 50 caracteres (como números de página) 
    return [p.strip() for p in paragrafos if len(p.strip()) > 50]


In [251]:
paragrafos = dividir_em_paragrafos(full_text)
print(f"\nParágrafos encontrados: {len(paragrafos)}")


Parágrafos encontrados: 2867


In [252]:
def chunk_text_especializado(paragrafos, max_chunk_chars=600, overlap_chars=50):
    """
    Agrupa parágrafos em blocos menores (chunks) com sobreposição.
    
    Sugestões aplicadas:
    - max_chunk_chars=600: Chunks menores tendem a ser mais focados.
    - overlap_chars: Evita a perda de contexto nas 'emendas' entre blocos.
    """
    chunks = []
    chunk_atual = ""

    for paragrafo in paragrafos:
        # TRATAMENTO DE INTEGRIDADE: 
        # Se um parágrafo sozinho for maior que o limite máximo, ele é dividido em partes.
        if len(paragrafo) > max_chunk_chars:
            if chunk_atual:
                chunks.append(chunk_atual)
                chunk_atual = ""
            # Divide o parágrafo gigante respeitando o overlap
            for i in range(0, len(paragrafo), max_chunk_chars - overlap_chars):
                chunks.append(paragrafo[i : i + max_chunk_chars])
            continue

        # Lógica de agrupamento com verificação de limite
        if len(chunk_atual) + len(paragrafo) + 1 <= max_chunk_chars:
            chunk_atual += (" " if chunk_atual else "") + paragrafo 
        else:
            if chunk_atual:
                chunks.append(chunk_atual)
            
            # APLICAÇÃO DE SOBREPOSIÇÃO (Sliding Window):
            # O novo chunk começa com o final do chunk anterior para manter o contexto [2].
            inicio_com_contexto = chunk_atual[-overlap_chars:] if len(chunk_atual) > overlap_chars else ""
            chunk_atual = (inicio_com_contexto + " " if inicio_com_contexto else "") + paragrafo

    if chunk_atual:
        chunks.append(chunk_atual)

    return chunks


In [253]:
chunks = chunk_text_especializado(paragrafos, max_chunk_chars=600)
print(f"Número de chunks: {len(chunks)}")
print(f"\nExemplo de chunk (primeiro):\n{chunks[0][:600]}...")
print(f"\nExemplo de chunk (segundo):\n{chunks[1][:600]}...")

Número de chunks: 456

Exemplo de chunk (primeiro):
Alésio dos Passos Santos – ambientalista, cultivador e educador de plantas medicinais (com contribuições de César Paulo Simionatto, Murilo Leandro Marcos, Leila Nery dos Santos Souza, Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon O sinergismo é uma das coisas mais importantes no mundo das plantas. O remédio de farmácia, o que que é? Um princípio ativo isolado, concentrado, sintetizado, que faz o remédio. E a planta, o que que faz? Todos os princípios ativos juntos em sinergismo, trabalhando juntos, é a energia da planta que não se...

Exemplo de chunk (segundo):
abalhando juntos, é a energia da planta que não se vê em microscópio. Então pra mim, isolar o princípio ativo é perder o sinergismo da planta. O remédio do sinergismo é das coisas mais importantes nas plantas. Nunca vi livro com esse Não adianta estudar somente os grupos de princípios ativos. A planta tem que ter a energia de cura. E a energia de cura só as pessoa

Carregando modelo

In [219]:
# Alternativa para GPU com pouca VRAM
from transformers import BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-3B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=HF_TOKEN,
    quantization_config=bnb_config,  # substitui o torch_dtype
    device_map="auto"
)


c:\Users\rayco\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rayco\.cache\huggingface\hub\models--meta-llama--Llama-3.2-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]c:\Users\rayco\AppData\Lo

In [263]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def is_bad_chunk(text, min_len=80):
    text = text.strip()
    if len(text) < min_len:
        return True
    words = text.split()
    if len(set(words)) < 10:
        return True
    strange_ratio = sum(1 for c in text if not c.isalnum() and c not in " .,;:-()[]{}") / max(len(text), 1) #
    if strange_ratio > 0.3:
        return True
    return False

def run_model_llama(model, tokenizer, chunk):
    messages = build_messages(chunk)

    # apply_chat_template cuida de TODOS os tokens especiais do Llama 3.2
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,           # já tokeniza direto
        add_generation_prompt=True,
        return_dict=True,        # retorna dict com input_ids e attention_mask
        return_tensors="pt"
    ).to(model.device)

    # Define os terminadores corretos para o Llama 3.2
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            eos_token_id=terminators   # ← importante para o Llama 3.2 parar corretamente
        
        )

    # Retorna só o que o modelo gerou (sem o prompt)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

def extract_triple(generated_text):
    """
    Versão atualizada para capturar o formato textual PERGUNTA/RESPOSTA
    e estruturar no formato exigido pelo professor (Instruction / Output).
    """
    
    pergunta_match = re.search(r"PERGUNTA\s*:?\s*(.*?)(?=RESPOSTA\s*:|$)", generated_text, re.IGNORECASE | re.DOTALL) # O uso de re.DOTALL permite capturar múltiplas linhas na pergunta
    resposta_match = re.search(r"RESPOSTA\s*:?\s*(.*?)$", generated_text, re.IGNORECASE | re.DOTALL)

    if pergunta_match and resposta_match:
        return {
                "Instruction": pergunta_match.group(1).strip(),
                "Output": resposta_match.group(1).strip()
            }
        #pergunta = pergunta_match.group(1).strip()
        #resposta = resposta_match.group(1).strip()

        # Validação mínima de qualidade
        #if len(pergunta) > 10 and len(resposta) > 20:
            
    return None



In [264]:
def build_messages(chunk):
    return [
        {
            "role": "system",
            "content": (
                "Você é um especialista em plantas medicinais. "
                "Leia o texto e gere EXATAMENTE UMA pergunta e UMA resposta completa, detalhada e explicativa"
                "baseadas SOMENTE nas informações do texto. Não invente nada.\n\n"
                "Responda APENAS neste formato:\n"
                "PERGUNTA: ...\n"
                "RESPOSTA: ..."
            )
        },
        {
            "role": "user",
            "content": (
                "EXEMPLO:\n"
                "TEXTO: \"O alecrim possui propriedades antioxidantes e é usado "
                "em infusão para melhorar a memória.\"\n"
                "PERGUNTA: Quais são as propriedades medicinais do alecrim?\n"
                "RESPOSTA: O alecrim possui propriedades antioxidantes e é utilizado "
                "em infusão para melhorar a memória.\n\n"
                f"AGORA PROCESSE ESTE TEXTO:\n\"{chunk}\""
            )
        }
    ]

In [265]:
clean_chunks = [clean_text(c) for c in chunks if not is_bad_chunk(c)]
print(f"Chunks válidos: {len(clean_chunks)}")

Chunks válidos: 456


In [246]:
validos = 0
triplas_causal = []

# IMPORTANTE: Garanta que 'chunks' seja a sua variável fatiada com 400 caracteres
# Para o teste, pegamos os 5 primeiros
amostra_chunks = clean_chunks[:5]  

print("=== INICIANDO AMOSTRA DE TESTE COM LLAMA 3.2 ===")

for i, chunk in enumerate(amostra_chunks):
    
    # 1. Limpeza prévia do chunk usando função auxiliar
    chunk_limpo = clean_text(chunk)
    
    # 2. Filtro de segurança: pula o chunk se ele for considerado ruim/curto
    if is_bad_chunk(chunk_limpo, min_len=80):
        print(f"\n CHUNK {i}: Ignorado por não atingir os critérios mínimos de qualidade.")
        continue

    try:
        # 3. Execução direta usando nova função inteligente
        # Ela já aplica o chat_template, roda o modelo e extrai APENAS o texto da resposta
        generated = run_model_llama(model, tokenizer, chunk_limpo)

        print(f"\nCHUNK {i}")
        print("-" * 50)
        print(generated)  # Mostra o texto limpo gerado para você acompanhar
        print("-" * 50)

        # 4. Usa a sua nova função de extração via Regex (com suporte a re.DOTALL)
        triple = extract_triple(generated)

        # 5. Validação e salvamento
        if triple:
            triplas_causal.append(triple)
            validos += 1
            print("🟢 Formato Válido capturado e adicionado!")

        else:
            print("🔴 Falha no formato ou tamanho mínimo: O modelo falhou nas tags ou tamanho.")
            
    except Exception as e:
        print(f"⚠️ Erro ao processar o chunk {i}: {e}")

print("\n" + "=" * 50)
print(f"TESTE CONCLUÍDO | VÁLIDOS INTEGRADOS: {validos} de {len(amostra_chunks)}")
print("=" * 50)

[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


=== INICIANDO AMOSTRA DE TESTE COM LLAMA 3.2 ===


c:\Users\rayco\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



CHUNK 0
--------------------------------------------------
PERGUNTA: O que é um princípio ativo isolado, concentrado, sintetizado, em relação ao remédio de farmácia?
RESPOSTA: Um princípio ativo isolado, concentrado, sintetizado, é um princípio ativo que faz o remédio de farmácia.
--------------------------------------------------
🟢 Formato Válido capturado e adicionado!


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



CHUNK 1
--------------------------------------------------
PERGUNTA: Qual é a abordagem mais eficaz para estudar as plantas medicinais, de acordo com o texto?

RESPOSTA: De acordo com o texto, a abordagem mais eficaz para estudar as plantas medicinais é considerar a energia de cura da planta, que não pode ser isolada apenas com estudos sobre princípios ativos.
--------------------------------------------------
🟢 Formato Válido capturado e adicionado!


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



CHUNK 2
--------------------------------------------------
PERGUNTA: Como as plantas interagem entre as diferentes partes de suas estruturas e como elas lidam com os princípios ativos e as plantas companheiras e antagônicas?
--------------------------------------------------
🔴 Falha no formato ou tamanho mínimo: O modelo falhou nas tags ou tamanho.


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



CHUNK 3
--------------------------------------------------
PERGUNTA: Quais são as principais diferenças entre a abordagem tecnológica e a abordagem espiritual na medicina?
--------------------------------------------------
🔴 Falha no formato ou tamanho mínimo: O modelo falhou nas tags ou tamanho.

CHUNK 4
--------------------------------------------------
PERGUNTA: Quais são as principais dificuldades enfrentadas por profissionais de saúde ao lidar com doenças que afetam o corpo mental, emocional e espiritual?
--------------------------------------------------
🔴 Falha no formato ou tamanho mínimo: O modelo falhou nas tags ou tamanho.

TESTE CONCLUÍDO | VÁLIDOS INTEGRADOS: 2 de 5


In [266]:
triplas_causal = []

# amostra_chunks = clean_chunks[:30]  

print(f"=== INICIANDO GERAÇÃO DO DATASET OFICIAL COM 600 CHARS ({len(clean_chunks)} chunks) ===")

for i, chunk in enumerate(clean_chunks):
    
    # 1. Aplica a higienização de espaçamento do chunk antes de enviar ao modelo
    chunk_limpo = clean_text(chunk)
    
    # 2. Filtro de Segurança: Ignora chunks bizarros, muito curtos ou corrompidos do PDF
    if is_bad_chunk(chunk_limpo, min_len=80):
        print(f"⏭️ Chunk {i}: Ignorado por critérios de qualidade mínimos.")
        continue
        
    try:
        # 3. Executa o modelo usando a sua nova função robusta para o Llama 3.2
        # Ela já aplica o Chat Template interno, gerencia os tokens de parada e devolve só a resposta
        generated = run_model_llama(model, tokenizer, chunk_limpo)
            
        # 4. Extrai a tripla estruturada (Instruction / Output) usando o Regex atualizado com re.DOTALL
        triple = extract_triple(generated)
        
        # 5. Adiciona na lista se passou na validação de formato e comprimento mínimo interno
        if triple:  
            triplas_causal.append(triple)
            
        # Print de progresso a cada 40 chunks para você acompanhar o processamento
        if i % 40 == 0 and i > 0:
            print(f"Processados {i}/{len(clean_chunks)} chunks... Triplas válidas até agora: {len(triplas_causal)}")


        if len(triplas_causal) % 20 == 0:
            save_jsonl(triplas_causal, "backup_dataset.jsonl")
            
    except Exception as e:
        print(f"⚠️ Erro ao processar o chunk índice {i}: {e}")

print("\n" + "="*50)
print(f"🎉 FIM DO PROCESSAMENTO DO DATASET OFICIAL!")
print(f"Total de chunks processados: {len(clean_chunks)}")
print(f"Total de pares válidos gerados com sucesso: {len(triplas_causal)}")
print(f"Taxa de aproveitamento: {100*len(triplas_causal)/len(clean_chunks):.1f}%")
print("="*50)

[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


=== INICIANDO GERAÇÃO DO DATASET OFICIAL COM 600 CHARS (456 chunks) ===


c:\Users\rayco\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformer

⚠️ Erro ao processar o chunk índice 23: name 'save_jsonl' is not defined


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_toke

...

⚠️ Erro ao processar o chunk índice 43: name 'save_jsonl' is not defined


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_toke

⚠️ Erro ao processar o chunk índice 64: name 'save_jsonl' is not defined


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


KeyboardInterrupt: 

In [267]:
def clean_triple(triple):
    # 1. Remove espaços em branco das pontas de todas as chaves existentes
    for key in triple:
        triple[key] = triple[key].strip()
    
    # 2. Mantém a sua excelente regra de tamanho mínimo (com as chaves corrigidas)
    if len(triple["Output"]) < 20 or len(triple["Instruction"]) < 5:
        return None
        
    return triple

def clean_list(triple):
    # Filtra triplas inválidas após a limpeza
    return [t for t in (clean_triple(x) for x in triple) if t is not None]


# Executa a limpeza na sua lista final
triplas_causal = clean_list(triplas_causal)

print(f"Triplas causal após limpeza final: {len(triplas_causal)}")

Triplas causal após limpeza final: 61


In [259]:
def mostrar_exemplo(triplas, titulo, n=3):
    print(f"\n=== {titulo} ===")
    for i, t in enumerate(triplas[:n], 1):
        print(f"\n--- Exemplo {i} ---")
        print(json.dumps(t, indent=2, ensure_ascii=False))

mostrar_exemplo(triplas_causal, "meta-llama/Llama-3.2-3B-Instruct")


=== meta-llama/Llama-3.2-3B-Instruct ===

--- Exemplo 1 ---
{
  "Instruction": "Qual é a definição de sinergismo no contexto das plantas medicinais?",
  "Output": "O sinergismo é uma das coisas mais importantes no mundo das plantas, referindo-se à interação de todos os princípios ativos isolados, concentrados, sintetizados, que fazem o remédio, trabalhando juntos, e que é a energia da planta que não se pode descrever sem a presença de todos os princípios ativos."
}

--- Exemplo 2 ---
{
  "Instruction": "O que é considerado \"sinergismo\" na plantas e como isso se relaciona com a isolamento de princípios ativos?",
  "Output": "O sinergismo na planta se refere à energia de cura que não pode ser isolada apenas com princípios ativos, e que requer uma compreensão mais esotérica e intuitiva, que não é tão comum em estudos científicos."
}

--- Exemplo 3 ---
{
  "Instruction": "Como as plantas interagem entre as diferentes partes de suas estruturas e como elas lidam com os princípios ativos e

In [260]:
def salvar_jsonl(triplas, nome_arquivo):
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        for t in triplas:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"Dataset salvo em {nome_arquivo}")

salvar_jsonl(triplas_causal, "dataset_teste_final.jsonl")

Dataset salvo em dataset_teste_final.jsonl
